# Temporary signal-only diagnostics

Run all cells in a fresh kernel **before completing notebook 02**. Only the saved reference flow and signal-ratio ensemble are required. Keep the current TAG and run directory. This notebook never trains, renormalizes, selects members, or changes model files. Results go to `hybrid/signal_only_diagnostics/`.

We use one million fresh events per class (four times the previous diagnostic size), with fine bins around scores 0.70–0.85. Counts and approximate pointwise 95% intervals accompany the calibration curves. Exact-density comparisons test errors that can cancel inside calibration bins. Optional checkpoint comparisons investigate export or checkpoint-selection problems. Changing DIAGNOSTIC_SEED gives an independent repeat; finite-sample intervals do not include training uncertainty.


In [ ]:
# Run this cell first in each notebook. All five notebooks share RUN_NAME.
import os
import sys
import subprocess
from pathlib import Path

TAG = "sb10_exppoly_10m_slowlr_v3_20261001"  # Fresh run: 10M nominal-ratio events per class and gradual LR decay.
RUN_NAME = TAG
USE_DRIVE = True
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = Path("/content/nsbi_calibration_toolkit")
    BRANCH = "ml4hep_school_tutorial"
    if not (REPO / ".git").exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH,
            "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git", str(REPO),
        ], check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
        subprocess.run([
            "git", "-C", str(REPO), "sparse-checkout", "set", "src",
            "workshops/ml4hep_tifr_colab/calibration",
        ], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
    SOURCE = REPO / "workshops/ml4hep_tifr_colab/calibration"
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "-r",
        str(SOURCE / "requirements.txt"),
    ], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        RUN_BASE = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab/calibration_runs")
    else:
        RUN_BASE = Path("/content/calibration_runs")
else:
    # Open the notebook with calibration/ as its working directory.
    SOURCE = Path.cwd().resolve()
    REPO = SOURCE.parents[2]
    RUN_BASE = SOURCE / "runs"

sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(SOURCE))
os.chdir(SOURCE)
RUN = Path(os.environ.get("CALIBRATION_RUN", RUN_BASE / RUN_NAME)).resolve()
RUN.mkdir(parents=True, exist_ok=True)
print("Source:", SOURCE)
print("TAG:", TAG)
print("Run:", RUN)
if "CALIBRATION_RUN" in os.environ:
    print("CALIBRATION_RUN overrides the tagged path; verify that it points to the intended run.")
print("Use a fresh kernel when switching between the old exercises and calibration.")


In [ ]:
import json
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.special import expit, logsumexp
from IPython.display import display
from flow_reference import ReferenceFlow
from utils import RatioEnsemble
from sampling import sample_process, process_log_density
from ratio_diagnostics import diagnostic_report

N_PER_CLASS = 1_000_000
DIAGNOSTIC_SEED = 5_100_503
CHECK_EXPORT = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ROOT = RUN / 'hybrid'
SIGNAL = ROOT / 'ratios' / 'signal'
OUT = ROOT / 'signal_only_diagnostics' / f'seed_{DIAGNOSTIC_SEED}'
manifest = SIGNAL / 'ensemble.json'
if not manifest.is_file():
    raise FileNotFoundError(f'Completed signal ensemble required: {manifest}. No training started.')
metadata = json.loads(manifest.read_text())
members = metadata['members']
required = [ROOT / 'reference.pt']
for i in range(members):
    required += [SIGNAL / f'model{i}.onnx', SIGNAL / f'model_scaler{i}.bin']
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError('Missing saved files; no training started:\n' + '\n'.join(missing))
if N_PER_CLASS < 2:
    raise ValueError('Use at least two events per class.')
OUT.mkdir(parents=True, exist_ok=True)
flow = ReferenceFlow.load(ROOT / 'reference.pt', DEVICE)
ratio = RatioEnsemble.load(SIGNAL, device=DEVICE)
print('Saved ensemble metadata:')
display(metadata)
print('Diagnostic output:', OUT)


In [ ]:
# Fresh independent banks: no sample parquet or background model is needed.
numerator = sample_process('signal', N_PER_CLASS, np.random.default_rng(DIAGNOSTIC_SEED))
denominator = flow.sample(N_PER_CLASS, DIAGNOSTIC_SEED + 1, batch_size=8192)
logq_n = flow.log_prob(numerator, batch_size=8192).astype(np.float64)
logq_d = flow.log_prob(denominator, batch_size=8192).astype(np.float64)
exact_n = process_log_density(numerator, 'signal') - logq_n
exact_d = process_log_density(denominator, 'signal') - logq_d
member_n = ratio.member_log_ratios(numerator)
member_d = ratio.member_log_ratios(denominator)
ensemble_n = logsumexp(member_n, axis=0) - np.log(members)
ensemble_d = logsumexp(member_d, axis=0) - np.log(members)
# Retain global coverage, adding narrow fixed bins in the region of concern.
score_edges = np.unique(np.round(np.r_[np.linspace(0, 1, 26), np.arange(.70, .851, .01)], 8))
report = diagnostic_report(member_n, member_d, exact_n, exact_d,
                           title='Signal / saved reference; fresh events',
                           score_edges=score_edges)
display(report['summary'])
report['summary'].to_csv(OUT / 'summary.csv', index=False)
report['bins'].to_csv(OUT / 'bins.csv', index=False)
for name, fig in report['figures']:
    fig.savefig(OUT / f'{name}.png', dpi=150, bbox_inches='tight')
    display(fig)
    plt.close(fig)
for note in report['notes']:
    print(note)


## Resolve the feature around 0.75

The table and zoom use each model's own predicted-score bins, with counts retained even for sparse bins. The black ensemble curve uses the arithmetic mean of ratios. The exact-ratio curve is a useful finite-sample control, but its bins contain different events. The following common-region table instead selects **the same events using the ensemble score** and compares all members there.


In [ ]:
bins = report['bins']
focus = bins[(bins.diagnostic == 'calibration') & (bins.view == 'custom')
             & (bins.score_left >= .70) & (bins.score_right <= .85)]
display(focus[['model', 'score_left', 'score_right', 'n_num', 'n_den',
               'score_mean', 'fraction_num', 'residual_low95', 'residual_high95']])
fig, ax = plt.subplots(figsize=(9, 5))
for name, table in focus.groupby('model', sort=False):
    table = table[table.n_total > 0]
    color = 'black' if name == 'Ensemble' else 'darkorange' if name == 'Exact ratio' else None
    line, = ax.plot(table.score_mean, table.calibration_residual, '.-', label=name, color=color)
    ax.vlines(table.score_mean, table.residual_low95, table.residual_high95,
              color=line.get_color(), alpha=.45)
ax.axhline(0, color='grey', ls='--')
ax.set(xlim=(.70, .85), xlabel='Predicted score', ylabel='Observed fraction minus mean score',
       title='Tail calibration with pointwise approximate 95% intervals')
ax.legend()
fig.tight_layout()
fig.savefig(OUT / 'tail_zoom.png', dpi=150)
plt.show()
plt.close(fig)

# Same events for every model; distinguish mean bias from event-to-event error.
regions = [(.70, .75), (.75, .80), (.80, .85), (.85, 1.000001)]
models = [(f'Member {i+1}', member_n[i], member_d[i]) for i in range(members)]
models += [('Ensemble', ensemble_n, ensemble_d)]
rows = []
for lo, hi in regions:
    for label, selector, truth in [('numerator', ensemble_n, exact_n),
                                    ('denominator', ensemble_d, exact_d)]:
        mask = (expit(selector) >= lo) & (expit(selector) < hi)
        for name, ln, ld in models:
            values = (ln if label == 'numerator' else ld)[mask]
            error = values - truth[mask]
            oracle_residual = expit(truth[mask]) - expit(values)
            rows.append(dict(region=f'[{lo:.2f}, {min(hi,1):.2f})', sample=label, model=name,
                count=int(mask.sum()), fraction=mask.mean(),
                mean_log_error=error.mean() if len(error) else np.nan,
                mean_log_error_SE=error.std(ddof=1)/np.sqrt(len(error)) if len(error)>1 else np.nan,
                RMS_log_error=np.sqrt(np.mean(error**2)) if len(error) else np.nan,
                mean_exact_probability_minus_score=oracle_residual.mean() if len(error) else np.nan))
common_regions = pd.DataFrame(rows)
common_regions.to_csv(OUT / 'common_tail_regions.csv', index=False)
display(common_regions)


## Check ensemble sensitivity and extreme weights

Leave-one-member-out comparisons are exploratory; no member is removed from the saved ensemble. The normalization and weight-concentration summaries reveal whether a small number of reference events dominate the arithmetic mean. The event table lists the 30 largest ensemble weights on the fresh denominator bank for inspection, including each member's log ratio and the exact value.


In [ ]:
variants = models + [('Exact ratio', exact_n, exact_d)]
if members > 1:
    for i in range(members):
        keep = np.arange(members) != i
        variants.append((f'Without member {i+1}',
                         logsumexp(member_n[keep], axis=0)-np.log(members-1),
                         logsumexp(member_d[keep], axis=0)-np.log(members-1)))
baseline_n, baseline_d = np.logaddexp(0., -ensemble_n), np.logaddexp(0., ensemble_d)
sensitivity = []
for name, ln, ld in variants:
    loss_n, loss_d = np.logaddexp(0., -ln), np.logaddexp(0., ld)
    dn, dd = loss_n-baseline_n, loss_d-baseline_d
    scaled = np.exp(ld-ld.max())
    order = np.sort(scaled)[::-1]
    log_mean = logsumexp(ld)-np.log(len(ld))
    with np.errstate(over='ignore'):
        mean_ratio = np.exp(log_mean)
    sensitivity.append(dict(model=name, BCE=.5*(loss_n.mean()+loss_d.mean()),
        BCE_minus_ensemble=.5*(dn.mean()+dd.mean()),
        paired_difference_SE=.5*np.sqrt(dn.var(ddof=1)/len(dn)+dd.var(ddof=1)/len(dd)),
        mean_ratio_den=mean_ratio, log_mean_ratio_den=log_mean,
        ESS=scaled.sum()**2/np.square(scaled).sum(),
        top1_weight_fraction=order[:1].sum()/order.sum(),
        top10_weight_fraction=order[:10].sum()/order.sum(),
        top100_weight_fraction=order[:100].sum()/order.sum()))
sensitivity = pd.DataFrame(sensitivity)
sensitivity.to_csv(OUT / 'ensemble_sensitivity.csv', index=False)
display(sensitivity)
idx = np.argsort(ensemble_d)[-30:][::-1]
outliers = pd.DataFrame(denominator[idx], columns=[f'x{i+1}' for i in range(5)])
outliers['log_q'] = logq_d[idx]
outliers['exact_log_ratio'] = exact_d[idx]
outliers['ensemble_log_ratio'] = ensemble_d[idx]
for i in range(members):
    outliers[f'member_{i+1}_log_ratio'] = member_d[i, idx]
outliers['largest_member_fraction_of_ratio_sum'] = np.exp(
    member_d[:, idx].max(axis=0)-logsumexp(member_d[:, idx], axis=0))
outliers.to_csv(OUT / 'largest_denominator_weights.csv', index=False)
display(outliers)


## Optional checkpoint/export audit

The existing trainer shares a checkpoint directory among members, so filenames alone do not identify the member. Compare each checkpoint with each exported member using that member's saved scaler. A matching pair should have small floating-point differences. A missing match can also mean the corresponding checkpoint was removed; it is not by itself evidence of a faulty export. Saved epoch numbers use Lightning's zero-based convention. Only load checkpoints produced by your own run.


In [ ]:
if CHECK_EXPORT:
    from nsbi_common_utils.lightning_tools import DensityRatioLightning
    paths = sorted((SIGNAL / 'checkpoints').glob('*.ckpt'))
    # Include independent bulk events and the high-weight reference events.
    probe = np.concatenate([numerator[:128], denominator[:128], denominator[idx]])
    exported = ratio.member_log_ratios(probe)
    audit, failures = [], []
    for path in paths:
        try:
            saved = torch.load(path, map_location='cpu', weights_only=False)
            epoch = saved.get('epoch')
            params = saved.get('hyper_parameters', {})
            checkpoint = DensityRatioLightning.load_from_checkpoint(path, map_location='cpu').eval()
            for i, (scaler, _) in enumerate(ratio.members):
                scaled_probe = scaler.transform(pd.DataFrame(probe, columns=[f'x{j+1}' for j in range(5)]))
                with torch.no_grad():
                    predicted = checkpoint(torch.as_tensor(np.asarray(scaled_probe), dtype=torch.float32)).numpy().reshape(-1)
                error = predicted-exported[i]
                audit.append(dict(checkpoint=path.name, epoch=epoch, exported_member=i+1,
                    max_abs_logit_difference=np.max(np.abs(error)),
                    RMS_logit_difference=np.sqrt(np.mean(error**2)),
                    initial_lr=params.get('learning_rate'),
                    decay_factor=params.get('callback_factor'),
                    decay_interval=params.get('callback_patience')))
            del checkpoint, saved
        except Exception as exc:
            failures.append(dict(checkpoint=path.name, error=str(exc)))
    if audit:
        audit = pd.DataFrame(audit)
        audit.to_csv(OUT / 'checkpoint_export_audit.csv', index=False)
        display(audit)
        print('Closest checkpoint for each exported member (diagnostic only):')
        display(audit.loc[audit.groupby('exported_member').RMS_logit_difference.idxmin()])
    else:
        print('No comparable checkpoints available; ONNX diagnostics above remain usable.')
    if failures:
        display(pd.DataFrame(failures))
        pd.DataFrame(failures).to_csv(OUT / 'checkpoint_audit_failures.csv', index=False)

(OUT / 'settings.json').write_text(json.dumps(dict(
    tag=TAG, run=str(RUN), seed=DIAGNOSTIC_SEED, n_per_class=N_PER_CLASS,
    ensemble_metadata=metadata, checkpoint_audit=CHECK_EXPORT,
    note='Read-only signal diagnostic; no training, normalization, or member selection.'), indent=2))
print('Finished. Commit this notebook with its outputs so we can inspect the result.')
